# 03 · p-adic Basics: Qₚ, balls, and Hensel lifting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mircus/padics/blob/master/notebooks/03_padic_basics.ipynb)

Core mathematical structures in **padic-ds**: p-adic fields, ultrametric topology, balls, and Hensel's lemma.

In [ ]:
# Run this cell to install padic-ds (needed on Colab or a fresh environment)
import importlib, subprocess, sys
if importlib.util.find_spec("padic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/Mircus/padics.git"])
    print("padic-ds installed")
else:
    print("padic already available")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from padic import QpContext, Qp, QpBall, padic_abs, padic_dist, digits_p_adic, hensel_lift_simple
print("padic-ds loaded ✓")

## 1 · p-adic representation

In [ ]:
ctx = QpContext(p=5, prec=8)
for (num, den, label) in [(75, 1, "75"), (5, 1, "5"), (1, 25, "1/25")]:
    x = Qp.from_rational(ctx, num, den)
    digs = digits_p_adic(x, 8)
    print(f"  {label:>6}  →  {x}"
          f"  digits={digs}  |·|_5={padic_abs(x):.6f}")

## 2 · Ultrametric triangle inequality

In [ ]:
ctx7 = QpContext(p=7, prec=10)
triples = [(7, 49, 56), (1, 7, 8), (7, 14, 21)]

print(f"{'a':>5} {'b':>5} {'c':>5}   d(a,b)  d(b,c)  d(a,c)  ultra?")
print("-" * 56)
for (na, nb_, nc) in triples:
    xa = Qp.from_int(ctx7, na)
    xb = Qp.from_int(ctx7, nb_)
    xc = Qp.from_int(ctx7, nc)
    dab = padic_dist(xa, xb); dbc = padic_dist(xb, xc); dac = padic_dist(xa, xc)
    ok = dac <= max(dab, dbc) + 1e-12
    print(f"{na:>5} {nb_:>5} {nc:>5}   {dab:.4f}  {dbc:.4f}  {dac:.4f}  {'✓' if ok else '✗'}")

## 3 · Nested p-adic balls (visualisation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: nested circles diagram
ax = axes[0]
radii  = [1.0, 0.2, 0.04, 0.008]
colors = ["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"]
labels_r = ["B(0,1)", "B(0,1/5)", "B(0,1/25)", "B(0,1/125)"]
for r, c, lb in zip(radii[::-1], colors[::-1], labels_r[::-1]):
    ax.add_patch(plt.Circle((0.5, 0.5), r * 0.45, color=c, alpha=0.6, label=lb))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Nested 5-adic balls B(0, 5^{-k})")
ax.legend(loc="upper right")

# Right: which integers fall in each ball?
ax = axes[1]
ctx5 = QpContext(p=5, prec=6)
ns = list(range(0, 30))
vals5 = [padic_abs(Qp.from_int(ctx5, n)) if n > 0 else 0.0 for n in ns]
bar_colors = ["#2c7bb6" if v <= 0.04 else "#abd9e9" if v <= 0.2 else "#d7191c" for v in vals5]
ax.bar(ns, vals5, color=bar_colors)
ax.axhline(0.04, color="black", linestyle="--", linewidth=0.8, label="r=1/25")
ax.axhline(0.2,  color="black", linestyle=":",  linewidth=0.8, label="r=1/5")
ax.set_xlabel("n"); ax.set_ylabel("|n|₅")
ax.set_title("5-adic absolute values for n = 0..29")
ax.legend()

plt.tight_layout()
plt.savefig("padic_balls.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved padic_balls.png")

## 4 · Hensel lifting: √6 in Z₅ and √2 in Z₇

In [ ]:
# √6 in Z_5: f(X)=X²-6, f(1)=1-6=-5≡0 mod 5, f'(1)=2, gcd(2,5)=1 → lifts
ctx5 = QpContext(p=5, prec=10)
r5 = hensel_lift_simple(ctx5, lambda a: a*a-6, lambda a: 2*a, a0_mod_p=1, target_prec=10)
print(f"√6 in Z_5 (10 digits): {r5}")
print(f"  Base-5 digits: {digits_p_adic(r5, 10)}")
print()

# √2 in Z_7: 3²=9≡2 mod 7, a₀=3
ctx7 = QpContext(p=7, prec=12)
r7 = hensel_lift_simple(ctx7, lambda a: a*a-2, lambda a: 2*a, a0_mod_p=3, target_prec=12)
print(f"√2 in Z_7 (12 digits): {r7}")
print(f"  Base-7 digits: {digits_p_adic(r7, 12)}")